In [5]:
import sys
!{sys.executable} -m pip install langchain-community==0.2.12

  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB 217.9 kB/s eta 0:00:11
    --------------------------------------- 0.0/2.3 MB 393.8 kB/s eta 0:00:06
   - -------------------------------------- 0.1/2.3 MB 585.1 kB/s eta 0:00:04
   ---- ----------------------------------- 0.2/2.3 MB 1.3 MB/s eta 0:00:02
   --------- ------------------------------ 0.5/2.3 MB 2.3 MB/s eta 0:00:01
   ------------------- -------------------- 1.1/2.3 MB 3.9 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.3 MB 6.2 MB/s eta 0:00:01
   ---------------------------------------- 2.3/2.3 MB 6.4 MB/s eta 0:00:00
Using cached langchain-0.2.17-

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-chroma 1.1.0 requires langchain-core<2.0.0,>=1.1.3, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.2.4 which is incompatible.
langchain-google-genai 4.1.2 requires langchain-core<2.0.0,>=1.2.2, but you have langchain-core 0.2.43 which is incompatible.
langchain-groq 1.1.2 requires langchain-core<2.0.0,>=1.2.8, but you have langchain-core 0.2.43 which is incompatible.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.2.43 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: 

In [6]:
!pip install duckduckgo-search


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: c:\Users\sushil\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [7]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools.ddg_search import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

c:\Users\sushil\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\_internal\_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `DuckDuckGoSearchAPIWrapper` to V2.
  warnings.warn(


In [8]:
load_dotenv()
llm=ChatGroq(model="llama-3.3-70b-versatile")


In [9]:
# Tools
search_tool = DuckDuckGoSearchRun(region="us-en")

@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}


@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"
    r = requests.get(url)
    return r.json()

In [10]:

# Make tool list
tools = [get_stock_price, search_tool, calculator]

# Make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)

In [11]:
# state
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [12]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

tool_node = ToolNode(tools)  # Executes tool calls

In [13]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [14]:
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")    

In [15]:
chatbot = graph.compile()

chatbot

TypeError: draw_mermaid() got an unexpected keyword argument 'frontmatter_config'

In [16]:

# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)

ImportError: cannot import name 'TracerSessionV1' from 'langchain_core.tracers.schemas' (c:\Users\sushil\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_core\tracers\schemas.py)

In [20]:
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)
# Hello! How can I assist you today?
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

ImportError: cannot import name 'TracerSessionV1' from 'langchain_core.tracers.schemas' (c:\Users\sushil\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_core\tracers\schemas.py)

In [18]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple")]})
print(out["messages"][-1].content)

ImportError: cannot import name 'TracerSessionV1' from 'langchain_core.tracers.schemas' (c:\Users\sushil\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_core\tracers\schemas.py)

In [21]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares?")]})
print(out["messages"][-1].content)

ImportError: cannot import name 'TracerSessionV1' from 'langchain_core.tracers.schemas' (c:\Users\sushil\AppData\Local\Programs\Python\Python312\Lib\site-packages\langchain_core\tracers\schemas.py)